FewShotChatMessagePromptTemplate

In [ ]:
# [목적] 작업별 입력과 답변 예시를 준비합니다.
# 예시들은 새 요청과 가장 비슷한 사례를 찾는 기준이 되며, 이후 프롬프트에 참고 자료로 들어갑니다.
examples = [
    {
        "instruction": "당신은 회의록 작성 전문가 입니다....",
        "input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다...",
        "answer": """..."""
    },
    {
        "instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의...",
        "answer": """문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서..."""
    },
{
    "instruction": "당신은 문장 교정 전문가 입니다. 다음 주어진 문장을 교정해 주세요",
    "input": "우리 회사는 새로운 마케팅 전략을 도입하려고 한다....",
    "answer": "본 회사는 새로운 마케팅 전략을 도입함으로써, ..."
},
]

In [ ]:
# [목적] 의미가 비슷한 예시를 고르고 대화형 Few-shot 프롬프트를 만듭니다.
# 임베딩과 Chroma가 요청과 가까운 예시를 찾으면, 선택된 예시가 다음 모델 요청의 참고 자료가 됩니다.
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.example_selectors import (
    SemanticSimilarityExampleSelector,
)
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

chroma = Chroma("fewshot_chat", OpenAIEmbeddings())

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{instruction}:\n{input}"),
        ("ai", "{answer}"),
    ]
)

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples, OpenAIEmbeddings(), chroma, k=1,
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
)

In [ ]:
# [목적] 새 요청을 넣어 가장 비슷한 예시가 실제로 선택되는지 확인합니다.
# 선택 결과를 먼저 보면, 모델 호출 전에 예시 선택이 의도대로 작동하는지 점검할 수 있습니다.
question = {
    "instruction": "회의록을 작성해 주세요",
    "input": "2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다. 이 회의에는 프로젝트 매니저인 최현수, 주요 개발자인 황지연, UI/UX 디자이너인 김태영이 참석했다. 회의의 주요 목적은 프로젝트의 현재 진행 상황을 검토하고, 다가오는 마일스톤에 대한 계획을 수립하는 것이었다. 각 팀원은 자신의 작업 영역에 대한 업데이트를 제공했고, 팀은 다음 주까지의 목표를 설정했다.",
}

example_selector.select_examples(question)

In [ ]:
# [목적] 선택된 예시와 새 요청을 하나의 최종 대화 프롬프트로 조합합니다.
# 시스템 역할, 참고 예시, 사용자 요청 순서로 구성해 모델이 답변 방식과 맥락을 함께 이해하게 합니다.
final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant.",
        ),
        few_shot_prompt,
        ("human", "{instruction}\n{input}"),
    ]
)

In [ ]:
# [목적] 최종 프롬프트를 모델에 보내고 답변이 생성되는 대로 화면에 출력합니다.
# ChatOpenAI는 모델 연결을 만들고, stream은 긴 답변도 완성까지 기다리지 않고 순서대로 보여 줍니다.
from langchain_openai import ChatOpenAI
from langchain_teddynote.messages import stream_response

llm = ChatOpenAI()

chain = final_prompt | llm  # 체인 생성

answer = chain.stream(question)  # 실행 및 결과 출력
stream_response(answer)